# Qualitative ranking comparison: PPR + Hybrid RAG vs. Hybrid RAG only

Same idea as Table 10 in the HippoRAG paper ("Ranking result examples ... on several path-finding multi-hop questions"): for a bridge question that requires connecting two independent facts through a shared entity, show what each method actually retrieves and at what rank -- not just the final answer text, but whether the bridging evidence surfaces near the top of the retrieved list at all.

**Question selection.** Mined from `result_precision_hybrid_ppr.txt` / `result_precision_hybrid.txt` (per-question ragas `context_precision_without_reference` scores over the same 86 questions in `data/eval_multihop_100_ppr.jsonl`, run through PPR+Hybrid and Hybrid-only respectively). Matching questions by text and sorting by `score_ppr - score_hybrid` among genuine `question_type: bridge` rows:

1. "Which companies provide Decoupling Diodes..." -- PPR=1.0000 vs Hybrid=0.5556 (delta +0.444) -- **already tested**: both pipelines' first gold-evidence chunk landed at rank 2, so the ragas precision gap came from the surrounding chunks' relevance, not a first-hit-rank difference.
2. This run: **"Which approved vendor for On/Off Valves from 2\" NB up to and including 24\" (Flanged, Fire-Safe To API-607/BS 6755) is also listed as an approved vendor for Pipeline Valves and Special Items?"** (`question_type: bridge`, `sampler: multiseed`, `n_hops: 2`) -- PPR+Hybrid scored **0.6046**, Hybrid-only scored **0.2500** (delta +0.355). Hybrid-only's low absolute score (0.25, vs. 0.56 for the first question) makes this a case where Hybrid RAG doesn't just rank the bridge lower -- it largely fails to retrieve it as useful context at all, which is the more decisive version of the comparison to look for.

The gold answer is: *"Perar (Italy) is an approved vendor for On/Off Valves from 2\" NB up to and including 24\" (Flanged, Fire-Safe To API-607/BS 6755) and is also listed as an approved vendor for Pipeline Valves and Special Items."* -- so **PERAR** is the bridging vendor entity, structurally the same shape as the paper's Chlorambucil example: two independently-stated facts about the same entity that only become useful once connected.

In [3]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from run_ppr_hybrid_question import run_ppr_hybrid
from run_hybrid_only_question import run_hybrid_only

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
QUESTION = (
    'Which approved vendor for On/Off Valves from 2" NB up to and including 24" '
    '(Flanged, Fire-Safe To API-607/BS 6755) is also listed as an approved vendor '
    'for Pipeline Valves and Special Items?'
)

REFERENCE_ANSWER = (
    'Perar (Italy) is an approved vendor for On/Off Valves from 2" NB up to and '
    'including 24" (Flanged, Fire-Safe To API-607/BS 6755) and is also listed as '
    'an approved vendor for Pipeline Valves and Special Items.'
)

# From data/eval_multihop_100_ppr.jsonl -- the row for this exact question.
GOLD_CHUNK_IDS = [
    "ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_11_Exhibit_I_Major_Subcontractors_and_Vendors_Rev_0_20241216_pdf_size_541323_page_28_text_chunk_1",
    "ROC_INPEX_ITT_Ichthys_Onshore_AGRU_Upgrade_and_CCS_EPC_Contract_WS33071728_ITT_Part_C_-_11_Exhibit_I_Major_Subcontractors_and_Vendors_Rev_0_20241216_pdf_size_541323_page_42_text_chunk_2",
]
GOLD_ENTITY_NAMES = [
    "Pipeline Valves and Special Items", "PERAR",
    'On/Off Valves From 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755)',
]
BRIDGE_ENTITY = "PERAR"  # the hub entity that satisfies both constraints

TOP_N_DISPLAY = 20  # matches FINAL_CONTEXT_CHUNKS in both runner scripts -- ragas scored the FULL
# 20-chunk list, so a diagnostic capped at 8 can hide exactly where two pipelines actually diverge.

## Run both pipelines on the same question

In [12]:
ppr_result = run_ppr_hybrid(QUESTION)
hybrid_result = run_hybrid_only(QUESTION)

print(f"PPR + Hybrid seed entities: {ppr_result['seeds']}")
print(f"PPR + Hybrid confidence: {ppr_result['answer'].confidence_score}")
print(f"Hybrid-only confidence:  {hybrid_result['answer'].confidence_score}")

PPR + Hybrid seed entities: ['Vendor', 'On/Off Valves From 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755)', 'Fire safe API 607 OR BS 6755', 'Pipeline Valves and Special Items']
PPR + Hybrid confidence: 0.95
Hybrid-only confidence:  0.95


## Build ranked-result tables

For each pipeline's final (reranked) chunk list, show rank, source file/page, whether the chunk is one of the dataset's gold-evidence chunks, and -- for PPR + Hybrid only -- the graph relation(s) asserted by that chunk (subject --label--> object). This is the direct analogue of Table 10's numbered ranking lists, just chunk-level instead of Wikipedia-passage-level.

In [17]:
def format_relations(chunk: dict) -> str:
    rels = chunk.get("asserted_relations") or []
    if not rels:
        return ""
    return " | ".join(
        f"{r.get('subject')} --{r.get('label')}--> {r.get('object')}" for r in rels
    )


def build_rank_rows(reranked_chunks: list, gold_chunk_ids: list, top_n: int) -> pd.DataFrame:
    rows = []
    for i, chunk in enumerate(reranked_chunks[:top_n], 1):
        content = (chunk.get("content") or "").strip().replace("\n", " ")
        rows.append({
            "rank": i,
            "gold": "GOLD" if chunk.get("chunk_id") in gold_chunk_ids else "",
            "file_name": chunk.get("file_name"),
            "page": chunk.get("page_number"),
            "asserted_relations": format_relations(chunk),
            "snippet": content[:160],
        })
    return pd.DataFrame(rows)


def first_gold_rank(reranked_chunks: list, gold_chunk_ids: list):
    for i, chunk in enumerate(reranked_chunks, 1):
        if chunk.get("chunk_id") in gold_chunk_ids:
            return i
    return None


def bridge_entity_rank(reranked_chunks: list, bridge_entity: str):
    """First rank at which a chunk asserts a relation naming the bridge entity
    as subject or object -- only meaningful for the PPR-sourced list, since
    Hybrid-only chunks never carry asserted_relations."""
    for i, chunk in enumerate(reranked_chunks, 1):
        for rel in chunk.get("asserted_relations") or []:
            if bridge_entity.lower() in (rel.get("subject", "") + rel.get("object", "")).lower():
                return i, rel
    return None, None


def all_gold_ranks(reranked_chunks: list, gold_chunk_ids: list) -> dict:
    """Rank (1-based) of EVERY gold chunk found, not just the first -- a tied
    first-rank between two pipelines doesn't mean they retrieved equally well;
    one may bury or drop the second piece of gold evidence while the other
    keeps both near the top. This is usually what actually explains a ragas
    context_precision_without_reference gap when the first-hit rank ties."""
    ranks = {}
    for i, chunk in enumerate(reranked_chunks, 1):
        cid = chunk.get("chunk_id")
        if cid in gold_chunk_ids:
            ranks[cid] = i
    return {cid: ranks.get(cid) for cid in gold_chunk_ids}

In [18]:
print("PPR + Hybrid RAG -- ranked context")
ppr_table = build_rank_rows(ppr_result["reranked_chunks"], GOLD_CHUNK_IDS, TOP_N_DISPLAY)
ppr_table

PPR + Hybrid RAG -- ranked context


,rank,gold,file_name,page,asserted_relations,snippet
0,1,GOLD,WS33071728_ITT Part C - 11_Exhibit I_Major Sub...,42,Pipeline Valves and Special Items --lists_as_a...,Pipeline Valves and Special Items Approved V...
1,2,GOLD,WS33071728_ITT Part C - 11_Exhibit I_Major Sub...,28,"On/Off Valves Above NB 24"" (Flanged, Fire-Safe...",Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
2,3,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,42,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
3,4,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,43,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
4,5,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,21,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
5,6,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,24,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
6,7,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,20,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
7,8,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,23,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...


In [19]:
print("Hybrid RAG only -- ranked context")
hybrid_table = build_rank_rows(hybrid_result["reranked_chunks"], GOLD_CHUNK_IDS, TOP_N_DISPLAY)
hybrid_table

Hybrid RAG only -- ranked context


,rank,gold,file_name,page,asserted_relations,snippet
0,1,GOLD,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,42,,Pipeline Valves and Special Items Approved V...
1,2,GOLD,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,28,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
2,3,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,42,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
3,4,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,43,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
4,5,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,21,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
5,6,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,24,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
6,7,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,20,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...
7,8,,ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and...,23,,Tender No: WS33071728 ICHTHYS ONSHORE AGRU UP...


## Rank-position summary

Computed directly from the tables above -- not hardcoded -- so re-running this notebook against a different question (swap `QUESTION`/`GOLD_CHUNK_IDS`/`BRIDGE_ENTITY` in the config cell) regenerates an accurate comparison automatically.

In [24]:
r_ppr = first_gold_rank(ppr_result["reranked_chunks"], GOLD_CHUNK_IDS)
r_hybrid = first_gold_rank(hybrid_result["reranked_chunks"], GOLD_CHUNK_IDS)
bridge_rank, bridge_rel = bridge_entity_rank(ppr_result["reranked_chunks"], BRIDGE_ENTITY)

ppr_all_ranks = all_gold_ranks(ppr_result["reranked_chunks"], GOLD_CHUNK_IDS)
hybrid_all_ranks = all_gold_ranks(hybrid_result["reranked_chunks"], GOLD_CHUNK_IDS)

def fmt_rank(r, total):
    return f"rank {r}" if r is not None else f"not found in top {total}"

print("=" * 100)
print("BRIDGE QUESTION RANKING SUMMARY")
print("=" * 100)
print(f"Question: {QUESTION}")
print(f"Bridge entity (gold): {BRIDGE_ENTITY}")
print()
print(f"First gold-evidence chunk -- PPR + Hybrid:  {fmt_rank(r_ppr, len(ppr_result['reranked_chunks']))}")
print(f"First gold-evidence chunk -- Hybrid only:    {fmt_rank(r_hybrid, len(hybrid_result['reranked_chunks']))}")
print()
print(f"All {len(GOLD_CHUNK_IDS)} gold chunks -- rank in PPR + Hybrid:  "
      f"{[fmt_rank(v, len(ppr_result['reranked_chunks'])) for v in ppr_all_ranks.values()]}")
print(f"All {len(GOLD_CHUNK_IDS)} gold chunks -- rank in Hybrid only:   "
      f"{[fmt_rank(v, len(hybrid_result['reranked_chunks'])) for v in hybrid_all_ranks.values()]}")
print("(If the first-hit ranks above tie but these lists differ, that's usually what actually drives a "
      "ragas context_precision_without_reference gap: one pipeline keeps BOTH pieces of bridging evidence "
      "near the top, the other buries or drops the second one, diluting precision with less-relevant filler.)")
print()
if bridge_rank is not None:
    print(f"PPR + Hybrid surfaces a relation naming '{BRIDGE_ENTITY}' at rank {bridge_rank}:")
    print(f"    {bridge_rel.get('subject')} --{bridge_rel.get('label')}--> {bridge_rel.get('object')}: "
          f"{bridge_rel.get('description')}")
    print("This is the graph relation that directly does the bridging -- PPR's hub-aware PageRank plus "
          "relation reranking surfaces it as retrieved evidence, not just as text the LLM happens to "
          "read past. Hybrid RAG has no equivalent -- it can only rank whole chunks by "
          "lexical/semantic similarity to the question, with no mechanism to score a chunk higher because "
          "it structurally connects two other retrieved facts.")
else:
    print(f"No relation naming '{BRIDGE_ENTITY}' was asserted by any chunk in PPR + Hybrid's top "
          f"{len(ppr_result['reranked_chunks'])} -- re-check BRIDGE_ENTITY spelling or widen TOP_N_DISPLAY.")

BRIDGE QUESTION RANKING SUMMARY
Question: Which approved vendor for On/Off Valves from 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755) is also listed as an approved vendor for Pipeline Valves and Special Items?
Bridge entity (gold): PERAR

First gold-evidence chunk -- PPR + Hybrid:  rank 1
First gold-evidence chunk -- Hybrid only:    rank 1

All 2 gold chunks -- rank in PPR + Hybrid:  ['rank 2', 'rank 1']
All 2 gold chunks -- rank in Hybrid only:   ['rank 2', 'rank 1']
(If the first-hit ranks above tie but these lists differ, that's usually what actually drives a ragas context_precision_without_reference gap: one pipeline keeps BOTH pieces of bridging evidence near the top, the other buries or drops the second one, diluting precision with less-relevant filler.)

PPR + Hybrid surfaces a relation naming 'PERAR' at rank 1:
    Pipeline Valves and Special Items --lists_as_approved_vendor--> PERAR: The section 'Pipeline Valves and Special Items' lists Perar as an appro

## Why did the ragas score differ if the gold-chunk ranks are identical?

Both pipelines placed the 2 gold chunks at the exact same ranks (`[2, 1]`) -- so `context_precision_without_reference`'s gap (0.6046 vs. 0.2500) must come from how the LLM judge scored the OTHER retrieved chunks, not from the bridging evidence's position.

**Important:** the original ragas scores were computed against the FULL reranked list (`FINAL_CONTEXT_CHUNKS = 20` in both `run_ppr_hybrid_question.py` and `run_hybrid_only_question.py`), not just the top 8. If `TOP_N_DISPLAY` was left at 8, ranks 1-8 can come out identical between the two pipelines (RRF fusion + cross-encoder reranking converging on the same top slots when Hybrid RAG's candidates dominate both fused pools) while ranks 9-20 -- where the real divergence must be, since that's the only remaining part of what ragas actually scored -- go unexamined. `TOP_N_DISPLAY` is now set to 20 to match `FINAL_CONTEXT_CHUNKS` so this diagnostic covers the same pool ragas judged.

As a cheap, non-LLM proxy for the judge's per-context relevance call (no extra API calls), flag every chunk that mentions any `GOLD_ENTITY_NAMES` keyword or `BRIDGE_ENTITY` -- a rough stand-in for "thematically connected to the bridge" vs. "off-topic noise". Also note chunks whose content is just page-header boilerplate (e.g. "Tender No: WS33071728 ... EXHIBIT I ..." with no body text) -- those are near-empty filler regardless of topic match, and a judge would likely mark them unhelpful even if they technically come from the right document.

In [25]:
def topical_overlap(chunk: dict) -> bool:
    content = (chunk.get("content") or "").lower()
    keywords = [BRIDGE_ENTITY.lower()] + [k.lower() for k in GOLD_ENTITY_NAMES]
    return any(k in content for k in keywords)


def is_boilerplate(chunk: dict) -> bool:
    """Heuristic: header-only chunks in this dataset are short and consist almost
    entirely of the repeated tender/exhibit header line, no substantive body text."""
    content = (chunk.get("content") or "").strip()
    return len(content) < 200 and "tender no" in content.lower()


def noise_report(reranked_chunks: list, gold_chunk_ids: list, top_n: int) -> None:
    on_topic, off_topic, boilerplate = 0, 0, 0
    for i, chunk in enumerate(reranked_chunks[:top_n], 1):
        is_gold = chunk.get("chunk_id") in gold_chunk_ids
        if is_gold:
            continue  # only judging the OTHER (non-gold) slots here
        boiler = is_boilerplate(chunk)
        topical = topical_overlap(chunk)
        if boiler:
            boilerplate += 1
        elif topical:
            on_topic += 1
        else:
            off_topic += 1
        tag = "boilerplate" if boiler else ("on-topic" if topical else "off-topic")
        print(f"    [rank {i}, {tag}] {chunk.get('file_name')} (p.{chunk.get('page_number')}): "
              f"{(chunk.get('content') or '').strip()[:140]}")
    total = on_topic + off_topic + boilerplate
    print(f"    -> {on_topic} on-topic / {off_topic} off-topic / {boilerplate} boilerplate among the {total} non-gold slots")


print(f"PPR + Hybrid -- non-gold slots (top {TOP_N_DISPLAY}):")
noise_report(ppr_result["reranked_chunks"], GOLD_CHUNK_IDS, TOP_N_DISPLAY)
print()
print(f"Hybrid only -- non-gold slots (top {TOP_N_DISPLAY}):")
noise_report(hybrid_result["reranked_chunks"], GOLD_CHUNK_IDS, TOP_N_DISPLAY)

PPR + Hybrid -- non-gold slots (top 8):
    [rank 3, on-topic] ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 11_Exhibit I_Major Subcontractors and Vendors_Rev.0_20241216.pdf (p.42): Tender No: WS33071728 
ICHTHYS ONSHORE AGRU UPGRADE AND CCS PROJECT EPC CONTRACT 
EXHIBIT I – MAJOR SUBCONTRACTORS AND VENDORS 
 
 
Page 42 
    [rank 4, off-topic] ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 11_Exhibit I_Major Subcontractors and Vendors_Rev.0_20241216.pdf (p.43): Tender No: WS33071728 
ICHTHYS ONSHORE AGRU UPGRADE AND CCS PROJECT EPC CONTRACT 
EXHIBIT I – MAJOR SUBCONTRACTORS AND VENDORS 
 
 
Page 43 
    [rank 5, on-topic] ROC_INPEX/ITT_Ichthys Onshore AGRU Upgrade and CCS EPC Contract/WS33071728_ITT Part C - 11_Exhibit I_Major Subcontractors and Vendors_Rev.0_20241216.pdf (p.21): Tender No: WS33071728 
ICHTHYS ONSHORE AGRU UPGRADE AND CCS PROJECT EPC CONTRACT 
EXHIBIT I – MAJOR SUBCONTRACTORS AND VENDOR

## Final answers side by side

In [21]:
print("=" * 100)
print("REFERENCE ANSWER")
print("=" * 100)
print(REFERENCE_ANSWER)
print()
print("=" * 100)
print(f"PPR + HYBRID RAG ANSWER (confidence={ppr_result['answer'].confidence_score})")
print("=" * 100)
print(ppr_result["answer"].answer)
print()
print("=" * 100)
print(f"HYBRID RAG ONLY ANSWER (confidence={hybrid_result['answer'].confidence_score})")
print("=" * 100)
print(hybrid_result["answer"].answer)

REFERENCE ANSWER
Perar (Italy) is an approved vendor for On/Off Valves from 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755) and is also listed as an approved vendor for Pipeline Valves and Special Items.

PPR + HYBRID RAG ANSWER (confidence=0.95)
The approved vendor for On/Off Valves from 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755) that is also listed as an approved vendor for Pipeline Valves and Special Items is Tormene-Valvitalia.

HYBRID RAG ONLY ANSWER (confidence=0.95)
The approved vendors for On/Off Valves from 2" NB up to and including 24" (Flanged, Fire-Safe To API-607/BS 6755) are Mokveld (Netherlands), Petrovalves (Italy), Grove (International), Perar (Italy), Starline (Italy), Flow Control Technology (FCT) Tyco (France), and Tormene-Valveitalia (Italy). The approved vendors for Pipeline Valves and Special Items include ATV (Italy), Bel Valves (UK), Cameron Ring-O (Italy), Oms Saleri (Italy), Perar (Italy), and Tormene-Valvitali

## Notes for reuse

To try a different bridge question, re-mine the candidate list:

```python
import re

def parse(fname):
    pairs, cur_q = [], None
    for line in open(fname):
        m = re.match(r'^\[\d+/\d+\]\s*(.*)', line.strip())
        if m:
            cur_q = m.group(1).strip()
        m2 = re.search(r'context_precision_without_reference=([0-9.]+)', line)
        if m2 and cur_q is not None:
            pairs.append((cur_q, float(m2.group(1))))
            cur_q = None
    return dict(pairs)

ppr_scores = parse("result_precision_hybrid_ppr.txt")
hybrid_scores = parse("result_precision_hybrid.txt")
common = set(ppr_scores) & set(hybrid_scores)
ranked = sorted(((ppr_scores[q] - hybrid_scores[q], q) for q in common), reverse=True)
```

then cross-reference the full question text against `data/eval_multihop_100_ppr.jsonl` (matching on the truncated prefix) to pull the matching row's `gold_chunk_ids` / `gold_entity_names` / `reference_answer` for the new `QUESTION`/`GOLD_CHUNK_IDS`/`BRIDGE_ENTITY` values above -- prefer `question_type: bridge` rows over timeline/date-arithmetic ones for a genuine Table-10-style comparison.